In [ ]:
print("Hello World")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%pip install pandas numpy tensorflow sklearn matplotlib transformers keras


In [ ]:
import keras
from keras.models import Sequential , Model
from keras.layers import Dense, Dropout, Flatten
from keras.applications import VGG16
from keras.preprocessing import image

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import random

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

Setting random environment

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [ ]:
train_dir = '../aircraft_damage_dataset_v1/train'
test_dir = '../aircraft_damage_dataset_v1/test'
valid_dir = '../aircraft_damage_dataset_v1/valid'

setting up directories for flow from dict func

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

rescaling image to be between 1 and 0 matrix type shit

In [ ]:
img_row, img_col = 224, 224 #VGG16 model standard
input_shape = (224, 224, 3) # VGG16 model standard RGB input shape
epochs = 5 # no of epochs
batch_size = 32 # batch_size is the number of training examples the model processes before updating its weight

the flow_from_directory() function is used to read images directly from the directory and generate batches of data that will be fed into the model for training.

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    batch_size=batch_size,
    target_size=(img_row,img_col),
    seed=seed,
    class_mode='binary',
    shuffle=True
)

In [ ]:
test_generator = test_datagen.flow_from_directory(
    test_dir,
    batch_size=batch_size,
    target_size=(img_row,img_col),
    seed=seed,
    class_mode='binary',
    shuffle=False
)

In [ ]:
valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    batch_size=batch_size,
    target_size=(img_row,img_col),
    seed=seed,
    class_mode='binary',
    shuffle=True
)

In [ ]:
base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)

In [ ]:
output = base.layers[-1].output
output = Flatten()(output)
base = Model(base.input, output)


In [ ]:
for layer in base.layers:
    layer.trainable = False
# freezing the layers of VGG16 so its weights doesn't get updates,
# we do it because VGG16 is already trained on internet level data and it know
# edges and all features so it is helpful

## Transfer learning 

In [ ]:
from tensorflow.keras.optimizers import Adam

In [ ]:
model = Sequential()
model.add(base)
model.add(Dense(512, activation='relu'))
model.add(Dropout(rate=0.3))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0002),loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

model with keras api (practice)

In [ ]:
input = base.output
hl_1 = Dense(512, activation='relu')(input)
dl_1 = Dropout(rate=0.3)(hl_1)
hl_2 = Dense(512, activation='relu')(dl_1)
dl_2 = Dropout(rate=0.5)(hl_2)
op = Dense(1, activation='sigmoid')(dl_2)

In [ ]:
modelAPI = Model(inputs=base.inputs,outputs=op)

In [ ]:
modelAPI.compile(optimizer=Adam(learning_rate=0.0001),loss='binary_crossentropy',metrics=['accuracy'])
modelAPI.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

Stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = modelAPI.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=20,
    callbacks=[Stop]
    )

In [ ]:
train_history = modelAPI.history.history 

In [ ]:
plt.ylabel("Loss")
plt.xlabel('Epoch')
plt.plot(train_history['loss'])
plt.show()

plt.title("Validation Loss")
plt.ylabel("Loss")
plt.xlabel('Epoch')
plt.plot(train_history['val_loss'])
plt.show()

In [ ]:
Stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=20,
    callbacks=[Stop]
    )

In [ ]:
loss, accuracy = modelAPI.evaluate(test_generator)

print("Loss :", loss)
print("Accuracy :", accuracy)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
pred_probs = modelAPI.predict(test_generator)
pred_classes = (pred_probs > 0.5).astype(int).flatten()
true_classes = test_generator.classes
class_names = list(test_generator.class_indices.keys())
print(classification_report(
    true_classes,
    pred_classes,
    target_names=class_names
))
cm = confusion_matrix(true_classes, pred_classes)
print(cm)

In [ ]:
import joblib

In [ ]:
joblib.dump(modelAPI,'./model/MODEL_api.pkl')

In [ ]:
modelAPI.save("./model/DentCrackModel.keras")